# 🚁 Мост ArduPilot ↔ Isaac Sim — запуск в 2 клика

Весь код живёт в [isaac_bridge.py](../isaac_bridge.py). Рабочая конфигурация от 2026-07-08 (первый успешный NAV_TAKEOFF 5м): lockstep + Фикс-12 (фреймы FLU→FRD) + Iris 1.5кг. Полётные задания — в соседнем notebook `flight_missions.ipynb`.

**Один раз на систему** (PowerShell от админа) — два правила firewall:
```powershell
New-NetFirewallRule -DisplayName "ArduPilot JSON UDP 9002" -Direction Inbound -Protocol UDP -LocalPort 9002 -Action Allow
New-NetFirewallRule -DisplayName "ArduPilot MAVLink UDP 14550" -Direction Inbound -Protocol UDP -LocalPort 14550 -Action Allow
```

**Обычный запуск**: Isaac Sim открыт со сценой → ячейка «Импорт» → «МОСТ» → «SITL» → диагностика (пакеты растут) → в `flight_missions.ipynb` писать задания.

**После краша**: Stop→Play в Isaac → `await reset_motors()` → `stop_sitl()` → `launch_sitl()` → заново задание.

## 1. Импорт

In [ ]:
import sys
if r"C:\VSCODE\ISAAC" not in sys.path:
    sys.path.append(r"C:\VSCODE\ISAAC")
import importlib
import isaac_bridge
importlib.reload(isaac_bridge)   # подхватывать правки модуля без перезапуска kernel
from isaac_bridge import *
print("isaac_bridge загружен")

## 2. МОСТ — поднять всё одной командой (9 шагов, безопасно перезапускать)

In [ ]:
await bridge_up()

## 3. SITL — запустить ArduPilot (откроется отдельное окно)

In [ ]:
launch_sitl()

## Диагностика (запускать сколько угодно: пакеты должны расти, dt≈4.17мс)

In [ ]:
await diag()

## Утилиты: сброс после Stop→Play / рестарт SITL

In [ ]:
await reset_motors()

In [ ]:
stop_sitl()

## Тест знаков (диагностика фреймов, БЕЗ SITL!)

Исторический инструмент, которым был найден корень «волчка» (мир Isaac: Y=West). Использовать при любых сомнениях в знаках после изменения сцены/модели. SITL должен быть ВЫКЛЮЧЕН (`stop_sitl()`).

In [ ]:
import asyncio

async def _srv(m1, m2, m3, m4):
    await execute_in_isaac("_wsl_servos[:] = [%d,%d,%d,%d] + [1000]*12" % (m1, m2, m3, m4))

async def _tl(cmd):
    await execute_in_isaac("import omni.timeline as _t; _t.get_timeline_interface().%s()" % cmd)

r1 = await execute_in_isaac("print(_ap_pkt_count)")
await asyncio.sleep(1.0)
r2 = await execute_in_isaac("print(_ap_pkt_count)")
if r1.get("output") != r2.get("output"):
    print("СТОП: SITL работает (пакеты идут). stop_sitl() и запусти тест снова.")
else:
    tests = [
        ("YAW+   (верх M1,M2 - 'yaw вправо, по часовой сверху')", (1700, 1700, 1300, 1300)),
        ("ROLL+  (верх M2,M3 - 'правый бок вниз')",               (1300, 1700, 1700, 1300)),
        ("PITCH+ (верх M1,M3 - 'нос вверх')",                     (1700, 1300, 1700, 1300)),
    ]
    print("Тест знаков: 3 пробы, каждая со сбросом сцены...")
    for name, pat in tests:
        await _srv(1000, 1000, 1000, 1000)
        await _tl("stop")
        await asyncio.sleep(0.7)
        await _tl("play")
        await asyncio.sleep(0.7)
        await _srv(1500, 1500, 1500, 1500)
        await asyncio.sleep(0.35)
        await _srv(*pat)
        await asyncio.sleep(0.3)
        r = await execute_in_isaac(
            "print(_wsl_log[-1]['gyro_deg'], '| tilt', _wsl_log[-1]['tilt_deg'], 'deg | alt', _wsl_log[-1]['alt'])"
        )
        await _srv(1000, 1000, 1000, 1000)
        print()
        print(name)
        print("   gyro тела [x,y,z] град/с (фрейм z-up):", r.get("output", "").strip())
    await _tl("stop")
    await asyncio.sleep(0.5)
    await _tl("play")
    await _srv(1000, 1000, 1000, 1000)
    print()
    print("Готово.")